In [64]:
import pandas as pd
import sqlite3

df = pd.read_csv('data/loan.csv')
conn = sqlite3.connect('loans.db')
df.to_sql('loans', conn, if_exists='replace', index=False)

/var/folders/_8/zh8sf2z52c738rdw5xv8jvsh0000gp/T/ipykernel_67873/3750938750.py:4: DtypeWarning: Columns (0: desc, 1: next_pymnt_d, 2: verification_status_joint, 3: sec_app_earliest_cr_line, 4: hardship_type, 5: hardship_reason, 6: hardship_status, 7: hardship_start_date, 8: hardship_end_date, 9: payment_plan_start_date, 10: hardship_loan_status, 11: debt_settlement_flag_date, 12: settlement_status, 13: settlement_date) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/loan.csv')


2260668

In [65]:
query = """
SELECT loan_status, COUNT(*) as count
FROM loans
GROUP BY loan_status
ORDER BY count DESC
"""
print('Loan Status Summary')
pd.read_sql(query, conn)

Loan Status Summary


,loan_status,count
0,Fully Paid,1041952
1,Current,919695
2,Charged Off,261655
3,Late (31-120 days),21897
4,In Grace Period,8952
5,Late (16-30 days),3737
6,Does not meet the credit policy. Status:Fully ...,1988
7,Does not meet the credit policy. Status:Charge...,761
8,Default,31


In [79]:
# missing = df.isnull().sum().sort_values(ascending=False)
# missing_pct = (missing / len(df) * 100).round(2)
# pd.DataFrame({'missing count': missing, 'missing_pct': missing_pct}).head(20)

# df = df.drop(columns=['id', 'url', 'member_id'])

missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing count': missing, 'missing_pct': missing_pct}).head(20)

,missing count,missing_pct
orig_projected_additional_accrued_interest,2252242,99.63
hardship_amount,2250055,99.53
hardship_length,2250055,99.53
hardship_type,2250055,99.53
hardship_reason,2250055,99.53
hardship_status,2250055,99.53
deferral_term,2250055,99.53
hardship_start_date,2250055,99.53
hardship_last_payment_amount,2250055,99.53
hardship_end_date,2250055,99.53


In [67]:
query = """
SELECT grade, COUNT(*) as total_loans, SUM(CASE 
    WHEN loan_status LIKE '%Charged off%' OR loan_status LIKE '%Default%' THEN 1 
    ELSE 0 END) as defaults,
    ROUND(100.0 * SUM(CASE
    WHEN loan_status LIKE '%Charged off%' OR loan_status LIKE '%Default%' THEN 1
    ELSE 0 END) / COUNT(*), 2) as default_rate_pct
FROM loans
GROUP BY grade
ORDER BY grade
"""
print('Default Rates by Loan Grade')
pd.read_sql(query, conn)

Default Rates by Loan Grade


,grade,total_loans,defaults,default_rate_pct
0,A,433027,13776,3.18
1,B,663557,51168,7.71
2,C,650053,83419,12.83
3,D,324424,59646,18.39
4,E,135639,35526,26.19
5,F,41800,14358,34.35
6,G,12168,4554,37.43


In [84]:
threshold = 0.5
cols_to_drop = missing_pct[missing_pct > threshold * 100].index.tolist()
print(f"Dropping {len(cols_to_drop)} columns:")
print(cols_to_drop)

df = df.drop(columns=cols_to_drop)
print(f"\nRemaining shape: {df.shape}")

Dropping 41 columns:
['orig_projected_additional_accrued_interest', 'hardship_amount', 'hardship_length', 'hardship_type', 'hardship_reason', 'hardship_status', 'deferral_term', 'hardship_start_date', 'hardship_last_payment_amount', 'hardship_end_date', 'payment_plan_start_date', 'hardship_payoff_balance_amount', 'hardship_loan_status', 'hardship_dpd', 'settlement_term', 'debt_settlement_flag_date', 'settlement_status', 'settlement_date', 'settlement_amount', 'settlement_percentage', 'sec_app_mths_since_last_major_derog', 'sec_app_revol_util', 'revol_bal_joint', 'sec_app_inq_last_6mths', 'sec_app_open_act_il', 'sec_app_num_rev_accts', 'sec_app_chargeoff_within_12_mths', 'sec_app_collections_12_mths_ex_med', 'sec_app_earliest_cr_line', 'sec_app_mort_acc', 'sec_app_open_acc', 'verification_status_joint', 'dti_joint', 'annual_inc_joint', 'desc', 'mths_since_last_record', 'mths_since_recent_bc_dlq', 'mths_since_last_major_derog', 'mths_since_recent_revol_delinq', 'next_pymnt_d', 'mths_sinc

In [37]:
query = """
SELECT *
FROM loans
LIMIT 1
"""
pd.read_sql(query, conn)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,None,None,2500,2500,2500.0,36 months,13.56,84.92,C,C1,...,None,None,Cash,N,None,None,None,None,None,None


In [39]:
pd.set_option('display.max_rows', None)
columns = pd.read_sql("PRAGMA table_info(loans)", conn)
columns

,cid,name,type,notnull,dflt_value,pk
0,0,id,REAL,0,None,0
1,1,member_id,REAL,0,None,0
2,2,loan_amnt,INTEGER,0,None,0
3,3,funded_amnt,INTEGER,0,None,0
4,4,funded_amnt_inv,REAL,0,None,0
5,5,term,TEXT,0,None,0
6,6,int_rate,REAL,0,None,0
7,7,installment,REAL,0,None,0
8,8,grade,TEXT,0,None,0
9,9,sub_grade,TEXT,0,None,0
